# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset. It uses `import json2vec as j2v`, `j2v.Address(...)`, and typed request objects instead of raw field dictionaries.

In [ ]:
import lightning.pytorch as lit
import torch
from rich.pretty import pprint

import json2vec as j2v

In [ ]:
@j2v.shim(yields=True)
def hello_world_records(observation: dict, strata: j2v.Strata):

    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
        {"color": "red", "label": "warm"},
        {"color": "blue", "label": "cool"},
    ]

    yield from records

In [ ]:
params = j2v.Hyperparameters(
    d_model=16,
    target=j2v.Address("record", "label"),
    embed=j2v.Address("record"),
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
)


model = j2v.JSON2Vec(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.StreamingDataModule.from_model(
    model,
    dataset=j2v.Dataset(processor=hello_world_records),
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    file_buffer_size=1,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-17 18:47:00.316 | INFO     | json2vec.architecture.root:__init__:128 - initialized JSON2Vec module


In [ ]:
trainer = lit.Trainer(accelerator="cpu", max_epochs=20, logger=False)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/grantham/Desktop/json2vec-oss/examples/checkpoints exists and is not empty.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃ In sizes ┃ Out sizes ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━┩
│ 0 │ nodes │ ModuleDict │ 14.5 K │ train │     0 │        ? │         ? │
└───┴───────┴────────────┴────────┴───────┴───────┴──────────┴───────────┘

Trainable params: 14.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 14.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 99                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 75.5 K

/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-05-17 18:47:00.374 | INFO     | json2vec.data.datasets:dataloader:670 - building dataloader
2026-05-17 18:47:00.374 | INFO     | json2vec.data.datasets:__init__:614 - initialized batch dataset


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_c
onnector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the
value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.

TypeError: hello_world_records() got multiple values for argument 'strata'

In [ ]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [ ]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [[0.9892922043800354], [0.9914584159851074]],
│   │   │   'null': [[0.0023610061034560204], [0.0021571198012679815]],
│   │   │   'padded': [[0.0029454415198415518], [0.0025102070067077875]],
│   │   │   'masked': [[0.0025872616097331047], [0.00227145291864872]],
│   │   │   'other': [[0.0028139660134911537], [0.0016026891535148025]]
│   │   },
│   │   'content': {
│   │   │   'value': [['warm'], ['cool']],
│   │   │   'probability': [[0.9768152236938477], [0.9677136540412903]],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   [
│   │   │   │   │   │   {'label': 'warm', 'probability': 0.9768152236938477},
│   │   │   │   │   │   {'label': 'cool', 'probability': 0.023184865713119507}
│   │   │   │   │   ]
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   [
│   │   │   │   │   │   {'label': 'cool', 'probability': 0.9677136540412903},
│   │   │   │   │   │   {'label': 'warm', 'probability': 0.032286420464515686}
│   │   │   │   │   ]
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [ ]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   [
│   │   │   │   │   0.37575697898864746,
│   │   │   │   │   -0.05500005930662155,
│   │   │   │   │   -0.03299124538898468,
│   │   │   │   │   0.1702968031167984,
│   │   │   │   │   0.31847241520881653,
│   │   │   │   │   -0.22782884538173676,
│   │   │   │   │   0.3130500614643097,
│   │   │   │   │   -0.19073443114757538,
│   │   │   │   │   -0.14237414300441742,
│   │   │   │   │   -0.10940338671207428,
│   │   │   │   │   -0.21174907684326172,
│   │   │   │   │   0.05635084956884384,
│   │   │   │   │   0.23270520567893982,
│   │   │   │   │   -0.20195645093917847,
│   │   │   │   │   0.3031195402145386,
│   │   │   │   │   -0.5204896926879883
│   │   │   │   ]
│   │   │   ],
│   │   │   [
│   │   │   │   [
│   │   │   │   │   -0.0023882128298282623,
│   │   │   │   │   -0.21255134046077728,
│   │   │   │   │   0.11988565325737,
│   │   │   │   │   -0.16955406963825226,
│   │   │   │   │   -0.29551148414611816,
│   │   │   │   │   -0.1711616814136505,
│   │   │   │   │   -0.13720552623271942,
│   │   │   │   │   0.49483561515808105,
│   │   │   │   │   0.30175966024398804,
│   │   │   │   │   0.2994464039802551,
│   │   │   │   │   0.09554266184568405,
│   │   │   │   │   -0.16718381643295288,
│   │   │   │   │   0.16135545074939728,
│   │   │   │   │   -0.14768348634243011,
│   │   │   │   │   -0.41188845038414,
│   │   │   │   │   0.3099677860736847
│   │   │   │   ]
│   │   │   ]
│   │   ]
│   }
}